In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import serial
import time

In [ ]:
# Loading the dataset (having moisture in %)
df = pd.read_csv("Soil_Quality_Converted.csv")
print("Sample Data:\n", df.head())

# Preprocess
X = df[['Temperature', 'Air Humidity', 'Moisture (%)']].values
y = df['Pump Data'].values  # Encode labels

scaler = StandardScaler()
X = scaler.fit_transform(X)

import joblib
joblib.dump(scaler, 'scaler.pkl')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Sample Data:
    Moisture (%)  Temperature  Air Humidity  Pump Data
0     53.084773    29.184908     71.789699          0
1     15.016814    33.707205     77.977391          1
2     49.666954    24.760311     60.776282          1
3     75.094020    32.738515     59.323543          0
4     15.795203    25.692744     66.624914          1


d:\New folder\venv\Lib\site-packages\keras\src\layers\core\dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/25
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.8946 - loss: 0.3197
Epoch 2/25
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9941 - loss: 0.0410
Epoch 3/25
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9976 - loss: 0.0248
Epoch 4/25
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9986 - loss: 0.0170
Epoch 5/25
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9934 - loss: 0.0196
Epoch 6/25
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9976 - loss: 0.0201
Epoch 7/25
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9989 - loss: 0.0124
Epoch 8/25
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.9962 - loss: 0.0320
Epoch 9/25
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9972 - loss: 0.0202
Epoch 10/25
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.9991 - loss: 0.0085
Epoch 11/25
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.9986 - loss: 0.0213
Epoch 12/25
1200/1200 ━━━━━━━━

✅ Model trained and saved as 'irrigation_model.h5'


In [ ]:
# Defining the Neural Network
model = Sequential([
    Dense(16, activation='relu', input_shape=(3,)),
    Dense(8, activation='relu'),
    Dense(1, activation='sigmoid')  # Binary classification
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(X_train, y_train, epochs=25, batch_size=2, verbose=1)               # Training

# Saving the Model
model.save('irrigation_model.h5')
print("Model trained and saved as 'irrigation_model.h5'")

In [ ]:
import serial
import time
import joblib
import numpy as np
from tensorflow.keras.models import load_model

# === Loading the ML model and scaler ===
model = load_model("irrigation_model.h5")
scaler = joblib.load("scaler.pkl")

# === Converting raw soil reading to moisture % based on calibration ===
def raw_to_percent(raw, dry=500, wet=2700):
    percent = (raw - dry) / (wet - dry) * 100
    return float(np.clip(percent, 0, 100))  # This is done to keep within 0–100%

# === Connect to ESP32 Serial Port ===
ser = serial.Serial('COM9', 9600, timeout=2)   #Change 'COM9' with ur port
time.sleep(2)
print("Connected to ESP32")

while True:
    try:
        # Reading a line from ESP32
        line = ser.readline().decode().strip()
        if not line:
            continue

        # Parsing values from ESP32
        temp, hum, soil_raw = map(float, line.split(","))
        moisture_percent = raw_to_percent(soil_raw)

        print(f" Temp:{temp:.2f}°C | Humidity:{hum:.2f}% | Moisture %:{moisture_percent:.2f}%")
        input_scaled = scaler.transform([[temp, hum, moisture_percent]])

        # Prediction
        prediction = model.predict(input_scaled)[0][0]
        status = "ON" if prediction >= 0.5 else "OFF"

        # Sending decision back to ESP32 
        ser.write((status + "\n").encode())
        print(f" Irrigation Command: {status} (Model Confidence: {prediction:.2f})\n")

    except Exception as e:
        print(" Error:", e)


✅ Connected to ESP32
 Temp:27.30°C | Humidity:95.00% | Moisture %:69.50%
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
 Irrigation Command: OFF (Model Confidence: 0.00)

 Temp:27.40°C | Humidity:95.00% | Moisture %:68.82%
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
 Irrigation Command: OFF (Model Confidence: 0.00)

 Temp:27.50°C | Humidity:95.00% | Moisture %:68.86%
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
 Irrigation Command: OFF (Model Confidence: 0.00)

 Temp:27.40°C | Humidity:95.00% | Moisture %:69.14%
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
 Irrigation Command: OFF (Model Confidence: 0.00)

 Temp:27.50°C | Humidity:95.00% | Moisture %:68.73%
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
 Irrigation Command: OFF (Model Confidence: 0.00)

 Temp:27.40°C | Humidity:95.00% | Moisture %:68.68%
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
 Irrigation Command: OFF (Model Confidence: 0.00)

 Temp:27.50°C | Humidity:95.00% | Moisture %:68.68%
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
 Irrigation Command: OFF (Model Confidence:

KeyboardInterrupt: 